# FLUX first-image emergency check
This notebook is a temporary manual recovery path. GPU availability and model are not guaranteed. Upload only the two-hour restore bundle generated locally.

In [ ]:
import json, os, pathlib, subprocess, sys
assert subprocess.run(['nvidia-smi'], check=False).returncode == 0, 'A CUDA GPU runtime is required'
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], check=True)
COMFY_COMMIT = 'da2608926eaf68fd532bba4e1ace3402c5d21399'
if not pathlib.Path('/content/ComfyUI').exists():
    subprocess.run(['git', 'clone', 'https://github.com/comfyanonymous/ComfyUI.git', '/content/ComfyUI'], check=True)
subprocess.run(['git', '-C', '/content/ComfyUI', 'checkout', '--detach', COMFY_COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', '/content/ComfyUI/requirements.txt'], check=True)


In [ ]:
from google.colab import files
import concurrent.futures, hashlib, urllib.request
uploaded = files.upload()
assert 'flux-first-image-colab-bundle.json' in uploaded
bundle = json.loads(uploaded['flux-first-image-colab-bundle.json'])
assert len(bundle['files']) == 3
def restore(item):
    target = pathlib.Path('/content/ComfyUI/models') / item['relative_path']
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = pathlib.Path(str(target) + '.part')
    start = partial.stat().st_size if partial.exists() else 0
    request = urllib.request.Request(item['download_url'], headers={'Range': f'bytes={start}-'} if start else {})
    with urllib.request.urlopen(request, timeout=120) as source, partial.open('ab' if start else 'wb') as output:
        while chunk := source.read(4 * 1024 * 1024): output.write(chunk)
    digest = hashlib.sha256()
    with partial.open('rb') as stream:
        while chunk := stream.read(4 * 1024 * 1024): digest.update(chunk)
    assert partial.stat().st_size == item['size_bytes'] and digest.hexdigest() == item['sha256']
    partial.replace(target)
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool: list(pool.map(restore, bundle['files']))


In [ ]:
import requests, time, uuid
server = subprocess.Popen([sys.executable, '/content/ComfyUI/main.py', '--listen', '127.0.0.1', '--port', '8188', '--user-directory', '/content/comfy-user', '--database-url', 'sqlite:////content/comfy-user/comfyui.db'])
for _ in range(120):
    try:
        if requests.get('http://127.0.0.1:8188/system_stats', timeout=2).ok: break
    except requests.RequestException: time.sleep(1)
workflow = {
 '1': {'class_type':'UNETLoader','inputs':{'unet_name':'flux-2-klein-4b-fp8.safetensors','weight_dtype':'default'}},
 '2': {'class_type':'CLIPLoader','inputs':{'clip_name':'qwen_3_4b.safetensors','type':'flux2','device':'default'}},
 '3': {'class_type':'CLIPTextEncode','inputs':{'text':'A cinematic wide shot of a futuristic white research station beside a clear blue ocean at sunset, realistic architecture, warm sunlight, detailed clouds, clean composition, high detail','clip':['2',0]}},
 '4': {'class_type':'FluxGuidance','inputs':{'conditioning':['3',0],'guidance':3.5}},
 '5': {'class_type':'EmptyFlux2LatentImage','inputs':{'width':1024,'height':1024,'batch_size':1}},
 '6': {'class_type':'KSampler','inputs':{'model':['1',0],'positive':['4',0],'negative':['4',0],'latent_image':['5',0],'seed':20260715,'steps':4,'cfg':1.0,'sampler_name':'euler','scheduler':'simple','denoise':1.0}},
 '7': {'class_type':'VAELoader','inputs':{'vae_name':'flux2-vae.safetensors'}},
 '8': {'class_type':'VAEDecode','inputs':{'samples':['6',0],'vae':['7',0]}},
 '9': {'class_type':'SaveImage','inputs':{'images':['8',0],'filename_prefix':'flux-first-image-colab'}}}
}
client_id = str(uuid.uuid4())
response = requests.post('http://127.0.0.1:8188/prompt', json={'prompt': workflow, 'client_id': client_id}, timeout=30)
response.raise_for_status(); prompt_id = response.json()['prompt_id']
for _ in range(900):
    history = requests.get(f'http://127.0.0.1:8188/history/{prompt_id}', timeout=10).json()
    if prompt_id in history: break
    time.sleep(2)
outputs = history[prompt_id]['outputs']; image = next(v['images'][0] for v in outputs.values() if v.get('images'))
png = requests.get('http://127.0.0.1:8188/view', params=image, timeout=120).content
pathlib.Path('/content/output.png').write_bytes(png)
assert png[:8] == bytes([137,80,78,71,13,10,26,10])
server.terminate(); files.download('/content/output.png')
